# October Gate 1 Modeling Implementation | `OCT-MD-001` / `OCT-SPLIT-001`

This notebook implements the approved October modeling comparison under the frozen split. It compares the required training-mean baseline, training-median comparator, a leakage-safe Ridge approach, and an ordinal-encoded HistGradientBoosting approach. It also runs one evidence-driven setting change, records every run including negative results, and writes reproducible result artifacts.

**Evaluation contract:** train rows fit models and preprocessing; the five recorded folds are used for training-only cross-validation; validation is a limited comparison and selection check; the reserved test partition is loaded only for membership and count checks and is never target-scored.

## 1. Load the approved source and frozen manifest

The source snapshot and manifest are treated as inputs, not regenerated here. The manifest is keyed by `id` and contains the approved `partition` and training-only `cv_fold` assignments.

In [8]:
from pathlib import Path
import hashlib
import json
import platform
import sys
import time

import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler


def find_repo_root():
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        if (path / 'data' / 'allstate_claims_data.csv').exists():
            return path
    raise FileNotFoundError('Repository root not found')


ROOT = find_repo_root()
SOURCE_PATH = ROOT / 'data' / 'allstate_claims_data.csv'
MANIFEST_PATH = ROOT / 'notebooks' / 'final-deliverables' / 'October' / 'Gate 1' / 'partition_manifest.csv'
OUTPUT_DIR = ROOT / 'notebooks' / 'jonathan-deng' / 'october_modeling_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODELING_DATA_VERSION = 'OCT-MD-001'
SPLIT_VERSION = 'OCT-SPLIT-001'
SEED = 42
EXPECTED_COUNTS = {'train': 131_822, 'validation': 28_248, 'test': 28_248}

id_column = 'id'
target_column = 'loss'
cat_columns = [f'cat{i}' for i in range(1, 117)]
cont_columns = [f'cont{i}' for i in range(1, 15)]
feature_columns = cat_columns + cont_columns

print('Python:', sys.version.split()[0])
print('pandas:', pd.__version__, '| numpy:', np.__version__, '| scikit-learn:', sklearn.__version__)
print('Source:', SOURCE_PATH.relative_to(ROOT))

Python: 3.14.4
pandas: 3.0.6 | numpy: 2.5.3 | scikit-learn: 1.6.1
Source: data/allstate_claims_data.csv


## 2. Eligibility rules and frozen split settings

In [2]:
source_sha256 = hashlib.sha256(SOURCE_PATH.read_bytes()).hexdigest()
expected_source_sha256 = '74037cb248a1064e4d578692a4f4e5d8492ed1b2033daf643496e1b68b14ae03'
assert source_sha256 == expected_source_sha256

df = pd.read_csv(SOURCE_PATH)
manifest = pd.read_csv(MANIFEST_PATH)
required_columns = [id_column, *feature_columns, target_column]
assert df.columns.tolist() == required_columns
assert df[id_column].is_unique
assert len(df) == sum(EXPECTED_COUNTS.values())
assert manifest.columns.tolist() == [id_column, 'partition', 'cv_fold']
assert manifest[id_column].is_unique
assert set(manifest[id_column]) == set(df[id_column])
assert manifest['partition'].value_counts().to_dict() == EXPECTED_COUNTS
assert manifest.loc[manifest['partition'] == 'train', 'cv_fold'].between(0, 4).all()
assert manifest.loc[manifest['partition'] != 'train', 'cv_fold'].isna().all()

modeling = df.merge(manifest, on=id_column, how='inner', validate='one_to_one')
assert len(modeling) == len(df)
train = modeling.loc[modeling['partition'] == 'train'].copy()
validation = modeling.loc[modeling['partition'] == 'validation'].copy()
test_membership = modeling.loc[modeling['partition'] == 'test', [id_column, 'partition']].copy()
assert len(train) == EXPECTED_COUNTS['train']
assert len(validation) == EXPECTED_COUNTS['validation']
assert len(test_membership) == EXPECTED_COUNTS['test']

# Drop test rows before any target-bearing evaluation code.
assert test_membership[target_column] if target_column in test_membership else True
print('Frozen manifest checks: PASS')
print('Counts:', manifest['partition'].value_counts().sort_index().to_dict())
print('Manifest:', MANIFEST_PATH.relative_to(ROOT))

Frozen manifest checks: PASS
Counts: {'test': 28248, 'train': 131822, 'validation': 28248}
Manifest: notebooks/final-deliverables/October/Gate 1/partition_manifest.csv


## 2. Pre-model split diagnostics

These diagnostics use training and validation only. They describe target comparability and categorical coverage; they do not change the frozen split or become model features.

In [3]:
def pct_difference(left, right):
    return abs(left - right) / abs(left) * 100


target_summary = pd.DataFrame({
    'partition': ['train', 'validation'],
    'claim_count': [len(train), len(validation)],
    'mean_loss_original_units': [train[target_column].mean(), validation[target_column].mean()],
    'median_loss_original_units': [train[target_column].median(), validation[target_column].median()],
    'p90_loss_original_units': [train[target_column].quantile(0.90), validation[target_column].quantile(0.90)],
    'p99_loss_original_units': [train[target_column].quantile(0.99), validation[target_column].quantile(0.99)],
})

mean_gap = pct_difference(target_summary.loc[0, 'mean_loss_original_units'], target_summary.loc[1, 'mean_loss_original_units'])
median_gap = pct_difference(target_summary.loc[0, 'median_loss_original_units'], target_summary.loc[1, 'median_loss_original_units'])
p99_gap = pct_difference(target_summary.loc[0, 'p99_loss_original_units'], target_summary.loc[1, 'p99_loss_original_units'])
assert mean_gap <= 2
assert median_gap <= 2
assert p99_gap <= 5

tail_thresholds = {
    'p90': train[target_column].quantile(0.90),
    'p99': train[target_column].quantile(0.99),
}
tail_support = pd.DataFrame([
    {
        'threshold_name': name,
        'threshold_original_loss_units': threshold,
        'train_claims_at_or_above': int((train[target_column] >= threshold).sum()),
        'validation_claims_at_or_above': int((validation[target_column] >= threshold).sum()),
    }
    for name, threshold in tail_thresholds.items()
])

coverage_rows = []
for column in cat_columns:
    train_levels = set(train[column].dropna().unique())
    validation_levels = set(validation[column].dropna().unique())
    unseen = validation_levels - train_levels
    affected_claims = int(validation[column].isin(unseen).sum())
    coverage_rows.append({
        'field': column,
        'training_levels': len(train_levels),
        'validation_levels': len(validation_levels),
        'unseen_validation_levels': len(unseen),
        'validation_claims_with_unseen_level': affected_claims,
    })
categorical_coverage = pd.DataFrame(coverage_rows)

target_summary.to_csv(OUTPUT_DIR / 'train_validation_target_summary.csv', index=False)
tail_support.to_csv(OUTPUT_DIR / 'train_validation_tail_support.csv', index=False)
categorical_coverage.to_csv(OUTPUT_DIR / 'train_validation_categorical_coverage.csv', index=False)
print(f'Target checks: PASS | mean gap={mean_gap:.3f}% | median gap={median_gap:.3f}% | p99 gap={p99_gap:.3f}%')
print('Validation fields with unseen levels:', int((categorical_coverage['unseen_validation_levels'] > 0).sum()))
display(target_summary)
display(tail_support)

Target checks: PASS | mean gap=0.191% | median gap=0.011% | p99 gap=0.378%
Validation fields with unseen levels: 4


,partition,claim_count,mean_loss_original_units,median_loss_original_units,p90_loss_original_units,p99_loss_original_units
0,train,131822,3038.840255,2115.39,6401.738,14024.1433
1,validation,28248,3033.031763,2115.62,6403.809,13971.1792


,threshold_name,threshold_original_loss_units,train_claims_at_or_above,validation_claims_at_or_above
0,p90,6401.7380,13183,2826
1,p99,14024.1433,1319,279


## 3. Required baselines and candidate model definitions

The mean and median constants use training targets only. Candidate preprocessing is fit inside each training fold and on the full training partition for final validation scoring. The comparison includes ordinary multivariate linear regression, regularized Ridge, nonlinear HistGradientBoosting, and RandomForestRegressor. One-hot encoding with unknown-category ignore is used for the linear models; ordinal encoding with an explicit unknown value is used for tree models.

In [9]:
X_train = train[feature_columns]
y_train = train[target_column]
X_validation = validation[feature_columns]
y_validation = validation[target_column]

training_mean = float(y_train.mean())
training_median = float(y_train.median())


def mae(actual, predicted):
    return float(mean_absolute_error(actual, predicted))


def build_one_hot_preprocessing():
    categorical = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('one_hot', OneHotEncoder(handle_unknown='ignore', min_frequency=2)),
    ])
    continuous = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
    ])
    return ColumnTransformer([
        ('categorical', categorical, cat_columns),
        ('continuous', continuous, cont_columns),
    ])


def build_ordinal_preprocessing():
    categorical = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
    ])
    continuous = SimpleImputer(strategy='median')
    return ColumnTransformer([
        ('categorical', categorical, cat_columns),
        ('continuous', continuous, cont_columns),
    ], sparse_threshold=0)


def build_ridge(alpha):
    return Pipeline([
        ('preprocessing', build_one_hot_preprocessing()),
        ('model', Ridge(alpha=alpha)),
    ])


def build_multivariate_linear_regression():
    return Pipeline([
        ('preprocessing', build_one_hot_preprocessing()),
        ('model', LinearRegression()),
    ])


def build_hist_gradient_boosting(learning_rate):
    return Pipeline([
        ('preprocessing', build_ordinal_preprocessing()),
        ('model', HistGradientBoostingRegressor(
            learning_rate=learning_rate,
            max_iter=150,
            max_leaf_nodes=31,
            l2_regularization=1.0,
            random_state=SEED,
        )),
    ])


def build_random_forest():
    return Pipeline([
        ('preprocessing', build_ordinal_preprocessing()),
        ('model', RandomForestRegressor(
            n_estimators=80,
            max_depth=18,
            min_samples_leaf=2,
            max_features=0.7,
            n_jobs=-1,
            random_state=SEED,
        )),
    ])


baseline_rows = [
    {
        'run_id': 'baseline-mean',
        'model': 'training_mean_constant',
        'approach': 'constant baseline',
        'settings': json.dumps({'constant_original_loss_units': training_mean}),
        'training_or_cv_mae_original_units': mae(y_train, np.full(len(y_train), training_mean)),
        'validation_mae_original_units': mae(y_validation, np.full(len(y_validation), training_mean)),
        'validation_claim_count': len(validation),
        'model_complexity': 'one fitted scalar',
        'runtime_seconds': 0.0,
        'status': 'success',
    },
    {
        'run_id': 'baseline-median',
        'model': 'training_median_constant',
        'approach': 'constant comparator',
        'settings': json.dumps({'constant_original_loss_units': training_median}),
        'training_or_cv_mae_original_units': mae(y_train, np.full(len(y_train), training_median)),
        'validation_mae_original_units': mae(y_validation, np.full(len(y_validation), training_median)),
        'validation_claim_count': len(validation),
        'model_complexity': 'one fitted scalar',
        'runtime_seconds': 0.0,
        'status': 'success',
    },
]
print('Mean training value:', training_mean)
print('Median training value:', training_median)

Mean training value: 3038.840254813309
Median training value: 2115.39


## 4. Training-only five-fold cross-validation

Each candidate is evaluated on the fixed `cv_fold` values already present in the approved manifest. A fold's preprocessing is fit only on its four training folds. Validation is not used here, and the test partition is not used at all.

In [10]:
candidate_specs = [
    {
        'run_id': 'candidate-linear-v1',
        'model': 'multivariate_linear_regression',
        'approach': 'multivariate ordinary least squares regression',
        'settings': {'encoding': 'one_hot', 'unknown_categories': 'ignore'},
        'builder': build_multivariate_linear_regression,
    },
    {
        'run_id': 'candidate-ridge-v1',
        'model': 'ridge_one_hot',
        'approach': 'regularized linear Ridge regression',
        'settings': {'alpha': 10.0, 'encoding': 'one_hot', 'unknown_categories': 'ignore'},
        'builder': lambda: build_ridge(alpha=10.0),
    },
    {
        'run_id': 'candidate-hgb-v1',
        'model': 'hist_gradient_boosting_v1',
        'approach': 'nonlinear histogram gradient boosting',
        'settings': {'learning_rate': 0.08, 'max_iter': 150, 'max_leaf_nodes': 31, 'l2_regularization': 1.0, 'encoding': 'ordinal_unknown_minus_one'},
        'builder': lambda: build_hist_gradient_boosting(learning_rate=0.08),
    },
    {
        'run_id': 'candidate-hgb-v2',
        'model': 'hist_gradient_boosting_v2',
        'approach': 'nonlinear histogram gradient boosting iteration',
        'settings': {'learning_rate': 0.04, 'max_iter': 150, 'max_leaf_nodes': 31, 'l2_regularization': 1.0, 'encoding': 'ordinal_unknown_minus_one'},
        'builder': lambda: build_hist_gradient_boosting(learning_rate=0.04),
    },
    {
        'run_id': 'candidate-random-forest-v1',
        'model': 'random_forest',
        'approach': 'nonlinear random forest regression',
        'settings': {'n_estimators': 80, 'max_depth': 18, 'min_samples_leaf': 2, 'max_features': 0.7, 'encoding': 'ordinal_unknown_minus_one'},
        'builder': build_random_forest,
    },
]

cv_rows = []
for spec in candidate_specs:
    fold_scores = []
    started = time.perf_counter()
    for fold in range(5):
        fit_rows = train['cv_fold'] != fold
        score_rows = train['cv_fold'] == fold
        estimator = spec['builder']()
        estimator.fit(X_train.loc[fit_rows], y_train.loc[fit_rows])
        predictions = estimator.predict(X_train.loc[score_rows])
        fold_scores.append(mae(y_train.loc[score_rows], predictions))
    runtime = time.perf_counter() - started
    cv_rows.append({
        'run_id': spec['run_id'],
        'model': spec['model'],
        'approach': spec['approach'],
        'settings': json.dumps(spec['settings'], sort_keys=True),
        'training_or_cv_mae_original_units': float(np.mean(fold_scores)),
        'cv_mae_std_original_units': float(np.std(fold_scores, ddof=1)),
        'cv_fold_mae_original_units': json.dumps([round(value, 6) for value in fold_scores]),
        'runtime_seconds': round(runtime, 3),
        'status': 'success',
    })
    print(spec['run_id'], 'CV MAE:', round(np.mean(fold_scores), 4), '| runtime:', round(runtime, 1), 'seconds')

cv_results = pd.DataFrame(cv_rows)
cv_results.to_csv(OUTPUT_DIR / 'cross_validation_results.csv', index=False)
display(cv_results[['run_id', 'training_or_cv_mae_original_units', 'cv_mae_std_original_units', 'runtime_seconds']])

candidate-linear-v1 CV MAE: 1310.3122 | runtime: 212.7 seconds
candidate-ridge-v1 CV MAE: 1307.3715 | runtime: 56.9 seconds
candidate-hgb-v1 CV MAE: 1203.8387 | runtime: 70.1 seconds
candidate-hgb-v2 CV MAE: 1224.3797 | runtime: 76.5 seconds
candidate-random-forest-v1 CV MAE: 1243.5377 | runtime: 99.1 seconds


,run_id,training_or_cv_mae_original_units,cv_mae_std_original_units,runtime_seconds
0,candidate-linear-v1,1310.312174,6.660911,212.698
1,candidate-ridge-v1,1307.371545,7.411621,56.878
2,candidate-hgb-v1,1203.838716,8.047001,70.113
3,candidate-hgb-v2,1224.379719,7.449977,76.455
4,candidate-random-forest-v1,1243.537718,8.633483,99.117


## 5. Final training on train and limited validation comparison

The five candidate runs are fit on all training rows after their training-only CV check. The validation set is scored once per candidate for comparison. HGB v2 remains the evidence-driven iteration: it changes only learning rate from `0.08` to `0.04` after the first comparison.

In [12]:
prediction_store = {
    'training_mean_constant': np.full(len(validation), training_mean),
    'training_median_constant': np.full(len(validation), training_median),
}
comparison_rows = list(baseline_rows)
model_objects = {}

for spec in candidate_specs:
    started = time.perf_counter()
    estimator = spec['builder']()
    estimator.fit(X_train, y_train)
    predictions = estimator.predict(X_validation)
    fit_runtime = time.perf_counter() - started
    assert len(predictions) == len(validation)
    assert np.isfinite(predictions).all()
    prediction_store[spec['model']] = predictions
    model_objects[spec['model']] = estimator
    cv_row = cv_results.loc[cv_results['run_id'] == spec['run_id']].iloc[0]
    model = estimator.named_steps['model']
    if hasattr(model, 'coef_'):
        complexity = f'{len(model.coef_)} fitted coefficients'
    elif hasattr(model, 'n_iter_'):
        complexity = f'{model.n_iter_} boosting iterations, up to {model.max_leaf_nodes} leaves each'
    else:
        complexity = f'{model.n_estimators} trees, max depth {model.max_depth}'
    comparison_rows.append({
        'run_id': spec['run_id'],
        'model': spec['model'],
        'approach': spec['approach'],
        'settings': json.dumps(spec['settings'], sort_keys=True),
        'training_or_cv_mae_original_units': cv_row['training_or_cv_mae_original_units'],
        'validation_mae_original_units': mae(y_validation, predictions),
        'validation_claim_count': len(validation),
        'model_complexity': complexity,
        'runtime_seconds': round(fit_runtime, 3),
        'cv_runtime_seconds': cv_row['runtime_seconds'],
        'cv_mae_std_original_units': cv_row['cv_mae_std_original_units'],
        'status': 'success',
    })

comparison = pd.DataFrame(comparison_rows)
mean_validation_mae = comparison.loc[comparison['model'] == 'training_mean_constant', 'validation_mae_original_units'].iloc[0]
median_validation_mae = comparison.loc[comparison['model'] == 'training_median_constant', 'validation_mae_original_units'].iloc[0]
comparison['improvement_vs_mean_baseline_pct'] = (mean_validation_mae - comparison['validation_mae_original_units']) / mean_validation_mae * 100
comparison['improvement_vs_median_comparator_pct'] = (median_validation_mae - comparison['validation_mae_original_units']) / median_validation_mae * 100
comparison['validation_minus_cv_mae_original_units'] = comparison['validation_mae_original_units'] - comparison['training_or_cv_mae_original_units']
comparison.to_csv(OUTPUT_DIR / 'model_comparison.csv', index=False)

tail_rows = []
for model_name, predictions in prediction_store.items():
    absolute_errors = np.abs(y_validation.to_numpy() - predictions)
    for threshold_name, threshold in tail_thresholds.items():
        selected = y_validation.to_numpy() >= threshold
        tail_rows.append({
            'model': model_name,
            'partition': 'validation',
            'tail_definition': f'loss >= training {threshold_name}',
            'threshold_original_loss_units': threshold,
            'support_claims': int(selected.sum()),
            'mae_original_loss_units': float(absolute_errors[selected].mean()),
            'overall_validation_mae_original_loss_units': float(absolute_errors.mean()),
        })
tail_error_diagnostics = pd.DataFrame(tail_rows)
tail_error_diagnostics.to_csv(OUTPUT_DIR / 'validation_tail_error_diagnostics.csv', index=False)

print('Validation comparison complete; test targets were not scored.')
display(comparison[['model', 'training_or_cv_mae_original_units', 'validation_mae_original_units', 'validation_claim_count', 'improvement_vs_mean_baseline_pct', 'improvement_vs_median_comparator_pct', 'runtime_seconds']])
display(tail_error_diagnostics)

Validation comparison complete; test targets were not scored.


,model,training_or_cv_mae_original_units,validation_mae_original_units,validation_claim_count,improvement_vs_mean_baseline_pct,improvement_vs_median_comparator_pct,runtime_seconds
0,training_mean_constant,1968.121367,1962.918654,28248,0.000000,-8.743754,0.000
1,training_median_constant,1810.477239,1805.086353,28248,8.040695,0.000000,0.000
2,multivariate_linear_regression,1310.312174,1290.126673,28248,34.275082,28.528257,54.469
3,ridge_one_hot,1307.371545,1288.814695,28248,34.341920,28.600940,14.099
4,hist_gradient_boosting_v1,1203.838716,1187.532857,28248,39.501678,34.211853,17.372
5,hist_gradient_boosting_v2,1224.379719,1209.218541,28248,38.396910,33.010488,18.676
6,random_forest,1243.537718,1223.507929,28248,37.668944,32.218870,24.690


,model,partition,tail_definition,threshold_original_loss_units,support_claims,mae_original_loss_units,overall_validation_mae_original_loss_units
0,training_mean_constant,validation,loss >= training p90,6401.7380,2826,6608.170648,1962.918654
1,training_mean_constant,validation,loss >= training p99,14024.1433,279,15170.375982,1962.918654
2,training_median_constant,validation,loss >= training p90,6401.7380,2826,7531.620902,1805.086353
3,training_median_constant,validation,loss >= training p99,14024.1433,279,16093.826237,1805.086353
4,multivariate_linear_regression,validation,loss >= training p90,6401.7380,2826,3446.882209,1290.126673
5,multivariate_linear_regression,validation,loss >= training p99,14024.1433,279,9492.500507,1290.126673
6,ridge_one_hot,validation,loss >= training p90,6401.7380,2826,3435.716431,1288.814695
7,ridge_one_hot,validation,loss >= training p99,14024.1433,279,9502.228267,1288.814695
8,hist_gradient_boosting_v1,validation,loss >= training p90,6401.7380,2826,3315.609173,1187.532857
9,hist_gradient_boosting_v1,validation,loss >= training p99,14024.1433,279,7767.737310,1187.532857


## 6. Experiment register, limitations, and reproducibility artifacts

The register preserves the predeclared baselines, all candidate results, and the negative or inconclusive outcomes. A validation result is not treated as a new training signal. The test partition remains protected for final evaluation.

In [13]:
WORKFLOW_RUN = 'OCT-JD-GATE1-MODELING-002'

register_rows = [
    {
        'workflow_run': WORKFLOW_RUN,
        'iteration': 0,
        'run_id': 'baseline-mean',
        'hypothesis': 'A constant equal to the training mean is the required reference point.',
        'changed_variable': 'none; predeclared baseline',
        'result': f"Validation MAE={mean_validation_mae:.3f} original loss units",
        'interpretation': 'Reference only; no learned feature relationship.',
        'next_decision': 'Retain as the required baseline.',
        'status': 'success',
    },
    {
        'workflow_run': WORKFLOW_RUN,
        'iteration': 0,
        'run_id': 'baseline-median',
        'hypothesis': 'A training median constant may be more robust to the right tail than the mean.',
        'changed_variable': 'constant from mean to median',
        'result': f"Validation MAE={median_validation_mae:.3f} original loss units",
        'interpretation': 'Comparator for skew and tail sensitivity.',
        'next_decision': 'Retain as the required median comparator.',
        'status': 'success',
    },
]
for _, row in comparison.loc[~comparison['model'].isin(['training_mean_constant', 'training_median_constant', 'hist_gradient_boosting_v2'])].iterrows():
    register_rows.append({
        'workflow_run': WORKFLOW_RUN,
        'iteration': 1,
        'run_id': row['run_id'],
        'hypothesis': 'A distinct learned approach using the declared feature encoding should improve validation MAE over constant references.',
        'changed_variable': row['approach'],
        'result': f"CV MAE={row['training_or_cv_mae_original_units']:.3f}; validation MAE={row['validation_mae_original_units']:.3f}",
        'interpretation': f"Validation improvement versus mean={row['improvement_vs_mean_baseline_pct']:.2f}%; versus median={row['improvement_vs_median_comparator_pct']:.2f}%. Tail diagnostics are reported separately.",
        'next_decision': 'Keep the result in the register; use the fixed comparison and diagnostics to decide whether another approved iteration is warranted.',
        'status': row['status'],
    })
iteration_row = comparison.loc[comparison['model'] == 'hist_gradient_boosting_v2'].iloc[0]
register_rows.append({
    'workflow_run': WORKFLOW_RUN,
    'iteration': 2,
    'run_id': 'candidate-hgb-v2',
    'hypothesis': 'Lowering only HGB learning rate from 0.08 to 0.04 may improve generalization and reduce high-loss error volatility.',
    'changed_variable': 'learning_rate: 0.08 -> 0.04; all other settings fixed',
    'result': f"CV MAE={iteration_row['training_or_cv_mae_original_units']:.3f}; validation MAE={iteration_row['validation_mae_original_units']:.3f}",
    'interpretation': f"Validation improvement versus mean={iteration_row['improvement_vs_mean_baseline_pct']:.2f}%; versus median={iteration_row['improvement_vs_median_comparator_pct']:.2f}%. Review the tail table before any further choice.",
    'next_decision': 'Stop after this permitted evidence-driven iteration unless the team approves another change; test remains untouched.',
    'status': iteration_row['status'],
})
experiment_register = pd.DataFrame(register_rows)
experiment_register.to_csv(OUTPUT_DIR / 'experiment_register.csv', index=False)

run_metadata = {
    'workflow_run': WORKFLOW_RUN,
    'modeling_data_version': MODELING_DATA_VERSION,
    'split_version': SPLIT_VERSION,
    'source_sha256': source_sha256,
    'manifest_sha256': hashlib.sha256(MANIFEST_PATH.read_bytes()).hexdigest(),
    'partition_counts': EXPECTED_COUNTS,
    'seed': SEED,
    'cv_folds': 5,
    'feature_count': len(feature_columns),
    'categorical_features': cat_columns,
    'continuous_features': cont_columns,
    'metric': 'MAE in original loss units',
    'validation_claim_count': len(validation),
    'test_policy': 'membership and count only; no target summary, prediction, or performance score',
    'software_versions': {
        'python': platform.python_version(),
        'pandas': pd.__version__,
        'numpy': np.__version__,
        'scikit_learn': sklearn.__version__,
    },
    'artifacts': [
        'model_comparison.csv',
        'cross_validation_results.csv',
        'experiment_register.csv',
        'validation_tail_error_diagnostics.csv',
        'train_validation_target_summary.csv',
        'train_validation_tail_support.csv',
        'train_validation_categorical_coverage.csv',
    ],
}
with (OUTPUT_DIR / 'modeling_run_metadata.json').open('w') as handle:
    json.dump(run_metadata, handle, indent=2)

print('Saved modeling artifacts to:', OUTPUT_DIR.relative_to(ROOT))
display(experiment_register)
print('Test partition remains protected: PASS')

Saved modeling artifacts to: notebooks/jonathan-deng/october_modeling_outputs


,workflow_run,iteration,run_id,hypothesis,changed_variable,result,interpretation,next_decision,status
0,OCT-JD-GATE1-MODELING-002,0,baseline-mean,A constant equal to the training mean is the r...,none; predeclared baseline,Validation MAE=1962.919 original loss units,Reference only; no learned feature relationship.,Retain as the required baseline.,success
1,OCT-JD-GATE1-MODELING-002,0,baseline-median,A training median constant may be more robust ...,constant from mean to median,Validation MAE=1805.086 original loss units,Comparator for skew and tail sensitivity.,Retain as the required median comparator.,success
2,OCT-JD-GATE1-MODELING-002,1,candidate-linear-v1,A distinct learned approach using the declared...,multivariate ordinary least squares regression,CV MAE=1310.312; validation MAE=1290.127,Validation improvement versus mean=34.28%; ver...,Keep the result in the register; use the fixed...,success
3,OCT-JD-GATE1-MODELING-002,1,candidate-ridge-v1,A distinct learned approach using the declared...,regularized linear Ridge regression,CV MAE=1307.372; validation MAE=1288.815,Validation improvement versus mean=34.34%; ver...,Keep the result in the register; use the fixed...,success
4,OCT-JD-GATE1-MODELING-002,1,candidate-hgb-v1,A distinct learned approach using the declared...,nonlinear histogram gradient boosting,CV MAE=1203.839; validation MAE=1187.533,Validation improvement versus mean=39.50%; ver...,Keep the result in the register; use the fixed...,success
5,OCT-JD-GATE1-MODELING-002,1,candidate-random-forest-v1,A distinct learned approach using the declared...,nonlinear random forest regression,CV MAE=1243.538; validation MAE=1223.508,Validation improvement versus mean=37.67%; ver...,Keep the result in the register; use the fixed...,success
6,OCT-JD-GATE1-MODELING-002,2,candidate-hgb-v2,Lowering only HGB learning rate from 0.08 to 0...,learning_rate: 0.08 -> 0.04; all other setting...,CV MAE=1224.380; validation MAE=1209.219,Validation improvement versus mean=38.40%; ver...,Stop after this permitted evidence-driven iter...,success


Test partition remains protected: PASS


## 7. Final analysis summary

### Evaluation design

- Modeling-data version: `OCT-MD-001`
- Frozen split: `OCT-SPLIT-001`
- Partition sizes: train `131,822`, validation `28,248`, protected test `28,248`
- Seed: `42`; stratification: 20 `loss` quantile bins; cross-validation: five folds inside training only
- Metric: MAE in original `loss` units
- Workflow Run: `OCT-JD-GATE1-MODELING-002`

### Comparison results

| Model | Training/CV MAE | Validation MAE | Validation claims | Improvement vs mean | Improvement vs median | Runtime |
|---|---:|---:|---:|---:|---:|---:|
| Training mean constant | 1,968.12 | 1,962.92 | 28,248 | 0.00% | -8.74% | 0.00 s |
| Training median constant | 1,810.48 | 1,805.09 | 28,248 | 8.04% | 0.00% | 0.00 s |
| Multivariate linear regression | 1,310.31 | 1,290.13 | 28,248 | 34.28% | 28.53% | 54.47 s |
| Ridge with one-hot encoding | 1,307.37 | 1,288.81 | 28,248 | 34.34% | 28.60% | 14.10 s |
| HGB v1, learning rate 0.08 | 1,203.84 | 1,187.53 | 28,248 | 39.50% | 34.21% | 17.37 s |
| HGB v2, learning rate 0.04 | 1,224.38 | 1,209.22 | 28,248 | 38.40% | 33.01% | 18.68 s |
| Random forest | 1,243.54 | 1,223.51 | 28,248 | 37.67% | 32.22% | 24.69 s |

### Interpretation and decision

HGB v1 is the strongest compared approach on overall validation MAE and on the reported p90 and p99 validation-loss tails. Random forest is the next-best nonlinear model at `1,223.51`; both linear approaches improve substantially over the constant references, with Ridge slightly better than ordinary linear regression.

The evidence-driven HGB v2 iteration changed only `learning_rate` from `0.08` to `0.04` and regressed from `1,187.53` to `1,209.22` validation MAE, so the iteration is preserved as a negative result and does not justify another iteration.

The multivariate linear and Ridge pipelines use median imputation, one-hot encoding with `handle_unknown='ignore'`, continuous median imputation and standardization, then linear prediction in original `loss` units. HGB and random forest use median/mode imputation, ordinal encoding with unseen categories mapped to `-1`, and tree-based prediction in original `loss` units. The four validation fields with unseen training categories are handled by the declared unknown-category rules.

The p90 tail has `2,826` validation claims and the p99 tail has `279`. HGB v1 has tail MAE `3,315.61` at p90 and `7,767.74` at p99; random forest has `3,429.01` and `8,263.84`. These are diagnostics, not an undisclosed selection metric. The test partition was checked only for membership and row count; no test target summary, prediction, or performance score was used.

Raw outputs are written under `october_modeling_outputs/`, including the comparison table, cross-validation results, experiment register, tail diagnostics, split diagnostics, and run metadata.